In [5]:
'''# Feature Engineering

Preparing customer data for churn prediction:
- Separate features and target
- Remove identifier columns
- Encode categorical variables
- Scale numerical variables
- Create a leakage-safe preprocessing pipeline'''

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from pathlib import Path

data_path = Path("../data/processed/telco_churn_cleaned.csv")
df = pd.read_csv(data_path)
print(df.shape)
df.head()

(7043, 26)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ExpectedCharges,ChargeRatio,Churn_numeric,HasPhoneService,HasMultipleLines
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,29.85,1.000000,0,0,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,1936.30,0.975830,0,1,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,107.70,1.004178,1,1,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,1903.50,0.967034,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,141.40,1.072489,1,1,0


In [6]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nChurn rate:")
print(y.mean().round(4))

Features shape: (7043, 24)
Target shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate:
0.2654


In [7]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print('training set shape:',x_train.shape)
print('test_set_shape:',x_test.shape)

print("\nTraining churn rate:", y_train.mean().round(4))
print("Testing churn rate:", y_test.mean().round(4))

training set shape: (5634, 24)
test_set_shape: (1409, 24)

Training churn rate: 0.2654
Testing churn rate: 0.2654


In [14]:
numerical_features=['tenure','MonthlyCharges','TotalCharges']
categorical_features=['gender','SeniorCitizen',"Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"]

print("Numeric features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numeric features: 3
Categorical features: 16
Total features: 19


In [15]:
numerical_transformer=Pipeline(steps=[('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_transformer=Pipeline(steps=[('imputer',SimpleImputer(strategy='most_frequent')),('Onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocessor=ColumnTransformer(transformers=[('num',numerical_transformer,numerical_features),('cat',categorical_transformer,categorical_features)])
print('preprocessing pipeline completed')

preprocessing pipeline completed


In [16]:
print("Actual DataFrame columns:")
print(X.columns.tolist())

print("\nCategorical features:")
print(categorical_features)

print("\nMissing feature names:")
print(set(categorical_features) - set(X.columns))

Actual DataFrame columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'ExpectedCharges', 'ChargeRatio', 'Churn_numeric', 'HasPhoneService', 'HasMultipleLines']

Categorical features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Missing feature names:
set()


In [17]:
x_train_processed=preprocessor.fit_transform(x_train)
x_test_processed=preprocessor.transform(x_test)

print(x_train_processed.shape)
print(x_test_processed.shape)

(5634, 46)
(1409, 46)


In [18]:
print("Training shape:", x_train_processed.shape)
print("Testing shape:", x_test_processed.shape)

print("\nMissing values in training:",
      np.isnan(x_train_processed).sum())

print("Missing values in testing:",
      np.isnan(x_test_processed).sum())

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training shape: (5634, 46)
Testing shape: (1409, 46)

Missing values in training: 0
Missing values in testing: 0

Training target distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [19]:
from pathlib import Path
import joblib

# Create artifact directory
artifact_dir = Path("../models/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    artifact_dir / "preprocessor.joblib"
)

# Save train/test datasets and targets
joblib.dump(
    {
        "X_train": x_train_processed,
        "X_test": x_test_processed,
        "y_train": y_train,
        "y_test": y_test
    },
    artifact_dir / "processed_data.joblib"
)

print("Preprocessing artifacts saved successfully!")

Preprocessing artifacts saved successfully!
